# Six-slot MRI training inputs

Attach completed tensor shards or notebook outputs. Images already contain the Raptor V5 crop, interpolation, intensity scaling and ImageNet normalization. The six-category extension and float16 storage are explicit in recipe.json.

This notebook demonstrates loading inputs, without training or changing your model. Gold74 remains held out. JEV FINAL2 supplies the optional soft training targets.


In [ ]:
"""Load exported tensors directly; performs no image preprocessing."""
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class ReadyTripletDataset(Dataset):
    """Use K=12 for fixed-size training bags, or k=None for full study bags.

Attach the preprocessing notebook outputs or published private shard datasets.
Images are already cropped, resized, intensity-scaled, and ImageNet-normalized.
The stored float16 tensor is suitable for training under torch.autocast.
"""
    def __init__(self, roots, *, split='train', k=12, labels=None):
        self.k = k
        if k is not None and (not isinstance(k, int) or k < 1):
            raise ValueError('k must be positive or None')
        roots = [Path(r) for r in roots]
        tables = [pd.read_csv(p, dtype={'StudyInstanceUID': str})
                  for root in roots for p in root.rglob('splits.csv')]
        if not tables:
            raise ValueError('No preprocessing splits.csv found')
        splits = pd.concat(tables, ignore_index=True)
        if splits.StudyInstanceUID.duplicated().any():
            raise ValueError('Attach each shard once; duplicate study IDs found')
        wanted = set(splits.loc[splits['split']==split, 'StudyInstanceUID'])
        self.files = sorted(p for root in roots for p in root.rglob('*.npz') if p.stem in wanted)
        if {p.stem for p in self.files} != wanted:
            raise ValueError('Attached shard metadata and tensor files disagree')
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file = self.files[index]
        with np.load(file, allow_pickle=False) as saved:
            x = torch.from_numpy(saved['images'])
            slots = torch.from_numpy(saved['triplet_slot_ids'])
            same_series = torch.from_numpy(saved['same_series'])
            present = torch.from_numpy(saved['slot_present'])
        if self.k is not None:
            indices = torch.randperm(len(x))[:self.k]
            if len(indices) < self.k:
                indices = torch.cat([indices, torch.randint(len(x), (self.k-len(indices),))])
            x, slots, same_series = x[indices], slots[indices], same_series[indices]
        item = {'images': x, 'triplet_slot_ids': slots, 'same_series': same_series,
                'slot_present': present, 'study_uid': file.stem}
        if self.labels is not None:
            item['targets'] = torch.as_tensor(self.labels[file.stem], dtype=torch.float32)
        return item


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

roots = []
for home in [Path('/kaggle/input'), Path('/kaggle/input/datasets/zafarnematov')]:
    roots.extend(home.glob('rsna-six-slot-raptor-v5-shard-*'))
for home in [Path('/kaggle/input'), Path('/kaggle/input/notebooks/zafarnematov')]:
    for notebook in home.glob('rsna-six-slot-raptor-v5-preprocessing-*'):
        roots.extend(notebook.glob('six_slot_raptor_shard_*'))
roots = sorted(set(p for p in roots if (p/'recipe.json').is_file()))
assert roots, 'Attach the completed preprocessing outputs or private tensor shard datasets'
recipes = [json.loads((p/'recipe.json').read_text()) for p in roots]
assert len({r['sha256'] for r in recipes}) == 1, 'Do not mix different preprocessing recipes'
assert all(r['source_budget'] == 64 and 'dicom_weight_type' in r.get('selector', '') for r in recipes)
assert all(json.loads((p/'manifest.json').read_text())['complete'] for p in roots)
print('Attached complete shards:', len(roots), '/ 32')


In [ ]:
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA',
          'Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
tables = [pd.read_csv(p/'labels_jev_final2.csv', dtype={'StudyInstanceUID':str}) for p in roots]
targets = pd.concat(tables, ignore_index=True)
assert not targets.StudyInstanceUID.duplicated().any()
assert np.isfinite(targets[LABELS].to_numpy()).all()
labels = dict(zip(targets.StudyInstanceUID, targets[LABELS].to_numpy()))
train = ReadyTripletDataset(roots, split='train', k=12, labels=labels)
loader = DataLoader(train, batch_size=2, shuffle=True, num_workers=2,
                    pin_memory=torch.cuda.is_available())
batch = next(iter(loader))
print('Training studies:', len(train))
print('Already normalized model inputs:', batch['images'].shape, batch['images'].dtype)
print('Soft targets:', batch['targets'].shape)
assert batch['images'].shape[1:] == (12,3,384,384)
assert torch.isfinite(batch['images']).all()
# The image tensor can now be passed directly to your Raptor-style model:
# with torch.autocast('cuda', dtype=torch.float16):
#     logits = model(batch['images'].cuda())
#     loss = torch.nn.functional.binary_cross_entropy_with_logits(
#         logits.float(), batch['targets'].cuda().float())


Use `batch["images"]` directly in your model. Avoid adding a second resize, `/255`, or normalization transform. `k=12` samples the exported bag; `k=None` loads its full set of windows. The six-element `slot_present` and per-channel `triplet_slot_ids` accompany the images.

All 32 shards are needed for the complete 4,407-study image corpus; a subset is useful for testing the training loop. Training uses the 4,333 non-gold studies when every shard is attached.
